In [ ]:
import pandas as pd
from pathlib import Path

# ============================================================
# Create pilot subset (100 images)


# Paths
BASE_DIR       = Path("..")
HAM_IMG_DIR    = BASE_DIR / "data" / "ham10000" / "images"
ISIC_IMG_DIR   = BASE_DIR / "data" / "isic2018" / "images"
HAM_CSV        = BASE_DIR / "data" / "preprocessed_manifests" / "ham10000_preprocessed.csv"
ISIC_CSV       = BASE_DIR / "data" / "preprocessed_manifests" / "isic2018_preprocessed.csv"
CHECKPOINT_DIR = Path("checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
PILOT_100_DIR = BASE_DIR / "data" / "pilot_100"
PILOT_100_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
PILOT_SIZE_HAM = 50
PILOT_SIZE_ISIC = 50


# ------------------------------------------------------------
# Load manifests

ham_df = pd.read_csv(HAM_CSV)

isic_df = pd.read_csv(ISIC_CSV)

ham_df["dataset"] = "HAM10000"
isic_df["dataset"] = "ISIC2018"

# ------------------------------------------------------------
# HAM sampling

ham_mel = ham_df[ham_df["label"] == 1]
ham_nonmel = ham_df[ham_df["label"] == 0]

ham_sample = pd.concat([
    ham_mel.sample(
        n=25,
        random_state=RANDOM_STATE
    ),
    ham_nonmel.sample(
        n=25,
        random_state=RANDOM_STATE
    )
])

# ------------------------------------------------------------
# ISIC sampling
# ------------------------------------------------------------

very_tiny = isic_df[isic_df["mask_size_class"] == "very_tiny"]

tiny = isic_df[isic_df["mask_size_class"] == "tiny"]

small = isic_df[isic_df["mask_size_class"] == "small"]

normal = isic_df[isic_df["mask_size_class"] == "normal"]

border_touching = isic_df[isic_df["touches_any_border"] == True]

isic_sample = pd.concat([
    very_tiny.sample(n=5, random_state=RANDOM_STATE),
    tiny.sample(n=5, random_state=RANDOM_STATE),
    small.sample(n=5, random_state=RANDOM_STATE),
    normal.sample(n=30, random_state=RANDOM_STATE),
    border_touching.sample(n=5, random_state=RANDOM_STATE), # may overlap with the above categories
])

# ------------------------------------------------------------
# Merge

pilot_df = pd.concat([
    ham_sample,
    isic_sample
]).reset_index(drop=True)

# ------------------------------------------------------------
# Save



pilot_csv = PILOT_100_DIR / "pilot_subset_100.csv"

pilot_df.to_csv(pilot_csv, index=False)

print("Saved:", pilot_csv)
print("Total images:", len(pilot_df))

display(
    pilot_df.groupby("dataset").size()
)